## README:

This notebook engineers predictive features on the preprocessed dataset.

**NOTE**: Cells labeled with '## ----- CONFIG ----- ##' contain parameters that need to be set manually 

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ta

from dotenv import load_dotenv

In [2]:
# add project root to path
root_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(root_path)

In [3]:
from data_processing.utils import get_indices_of_first_datapoint

## Load env variables

In [4]:
load_dotenv(r'../.env')

DATA_DIR = os.getenv('DATA_DIR')

## Global config

In [5]:
## ----- CONFIG ----- ##
WRITE_MODE = 'w' # 'x': do not overwrite

## Load data

In [6]:
data_df = pd.read_csv(os.path.join(DATA_DIR, 'preprocessed.csv'))
data_df.shape

(67630, 20)

In [7]:
data_df.columns

Index(['date', 'tic', 'close', 'high', 'low', 'open', 'volume', 'BAMLH0A0HYM2',
       'BOGMBASE', 'CPIAUCSL', 'DFF', 'ICSA', 'PPIACO', 'REAINTRATREARAT10Y',
       'T10Y2Y', 'USEPUINDXD', 'USREC', 'USSLIND', 'USSTHPI', 'VIXCLS'],
      dtype='object')

In [8]:
data_df.isna().any().any()

np.False_

In [9]:
data_df['date'] = pd.to_datetime(data_df['date'], format='%Y-%m-%d')
data_df = data_df.set_index('date')

## Time features

In [10]:
time_df = data_df.copy()

In [ ]:
## ----- CONFIG ----- ##
# time attributes
time_df['day_of_year'] = time_df.index.day_of_year.values - 1 # 1 to 365
time_df['month_of_year'] = time_df.index.month.values # 1 to 12
time_df['day_of_week'] = time_df.index.day_of_week.values + 1 # Monday: 1, Sunday: 7

# sine/cosine
time_df['doy_sin'] = np.sin(2 * np.pi * time_df['day_of_year'] / 365)
time_df['doy_cos'] = np.cos(2 * np.pi * time_df['day_of_year'] / 365)
time_df['month_sin'] = np.sin(2 * np.pi * time_df['month_of_year'] / 12)
time_df['month_cos'] = np.cos(2 * np.pi * time_df['month_of_year'] / 12)
time_df['dow_sin'] = np.sin(2 * np.pi * time_df['day_of_week'] / 5) # trading days
time_df['dow_cos'] = np.cos(2 * np.pi * time_df['day_of_week'] / 5) # trading days

In [12]:
# summary of time features
print( 'Unique days in a year:', time_df['day_of_year'].nunique() )
print( time_df['month_of_year'].value_counts(sort=False) )
print( time_df['day_of_week'].value_counts(sort=False) )

Unique days in a year: 365
month_of_year
1     5330
2     5180
3     5910
4     5580
5     5720
6     5730
7     5700
8     5990
9     5470
10    5960
11    5520
12    5540
Name: count, dtype: int64
day_of_week
5    13570
1    12690
2    13880
3    13870
4    13620
Name: count, dtype: int64


In [ ]:
# check for NAs
time_df.isna().any().any()

np.False_

## Technical indicators

In [14]:
## ----- CONFIG ----- ##
def add_indicators(grouped_data):
    """Add technical indicators to the stock prices.
    """
    # ATR: volatility
    atr = ta.volatility.AverageTrueRange(
        high=grouped_data['high'],
        low=grouped_data['low'],
        close=grouped_data['close'],
        window=30,
        fillna=False
    )
    grouped_data['atr_30'] = atr.average_true_range()

    # MACD: trend
    macd = ta.trend.MACD(grouped_data['close'])
    grouped_data['macd'] = macd.macd()
    grouped_data['macd_signal'] = macd.macd_signal()
    grouped_data['macd_hist'] = macd.macd_diff()

    # CCI: trend
    cci = ta.trend.CCIIndicator(
        high=grouped_data['high'], 
        low=grouped_data['low'], 
        close=grouped_data['close'], 
        window=30,
        fillna=False
    )
    grouped_data['cci_30'] = cci.cci()

    # RSI: momentum
    rsi = ta.momentum.RSIIndicator(
        grouped_data['close'], 
        window=30,
        fillna=False
    )
    grouped_data['rsi_30'] = rsi.rsi()

    return grouped_data

In [15]:
# apply indicators function
indicators_df = (
    time_df
    .groupby('tic')
    .apply(add_indicators, include_groups=False)
    .reset_index(level=0)
)
indicators_df.shape

(67630, 34)

In [16]:
na_df = indicators_df.isna()
na_perc = na_df.mean()
na_perc[na_perc > 0]

macd           0.003697
macd_signal    0.004879
macd_hist      0.004879
cci_30         0.004288
rsi_30         0.004288
dtype: float64

In [17]:
get_indices_of_first_datapoint(na_df)

,first datapoint,column
30,1999-03-11,macd_signal
31,1999-03-11,macd_hist
33,1999-03-05,rsi_30
32,1999-03-05,cci_30
29,1999-03-01,macd
5,1999-01-22,volume
0,1999-01-22,tic
4,1999-01-22,open
24,1999-01-22,month_sin
20,1999-01-22,month_of_year


In [18]:
## ----- CONFIG ----- ##
# choose a start date
START_DATE = '1999-03-11'

# choose feature to exclude
EXCLUDE_COLS = []

In [19]:
# subset columns
cols_filter = ~indicators_df.columns.isin(EXCLUDE_COLS)

# check NAs of subset
sub_df = indicators_df.loc[START_DATE:, cols_filter]
print('NAs detected:', na_df.loc[START_DATE:, cols_filter].any().any())
print('Data points per ticker:', sub_df.shape[0] / sub_df['tic'].nunique())

NAs detected: False
Data points per ticker: 6730.0


## Finalize and export

In [20]:
export_df = indicators_df.loc[START_DATE:, cols_filter].reset_index()

print(export_df.shape)
export_df.columns

(67300, 35)


Index(['date', 'tic', 'close', 'high', 'low', 'open', 'volume', 'BAMLH0A0HYM2',
       'BOGMBASE', 'CPIAUCSL', 'DFF', 'ICSA', 'PPIACO', 'REAINTRATREARAT10Y',
       'T10Y2Y', 'USEPUINDXD', 'USREC', 'USSLIND', 'USSTHPI', 'VIXCLS',
       'day_of_year', 'month_of_year', 'day_of_week', 'doy_sin', 'doy_cos',
       'month_sin', 'month_cos', 'dow_sin', 'dow_cos', 'atr_30', 'macd',
       'macd_signal', 'macd_hist', 'cci_30', 'rsi_30'],
      dtype='object')

In [21]:
# export data
out_path = os.path.join(DATA_DIR, 'data_with_features.csv')
export_df.to_csv(out_path, index=False, mode=WRITE_MODE)